# Imputation

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

## Experiment

Define output of experiment:

In [2]:
OUTPUTS = PROJECT_ROOT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

Import data, split into train/test and scale on train data.

In [3]:
from src.data import load_ett, chronological_split, scale

train, test = chronological_split(load_ett())
train_s, test_s, scaler = scale(train=train, test=test)

Set location for storing reconstructed datasets.

In [4]:
from src.experiments.artifacts import ReconstructedStore

store = ReconstructedStore(OUTPUTS / "reconstructed")

Initialise experiment.

In [5]:
from src.experiments.imputation import ImputationExperiment
from src.imputers import IMPUTERS

experiment = ImputationExperiment(
    mechanisms=["mar", "mcar", "mnar"],
    proportions=[10, 30, 50],
    seeds=range(10),
    imputer_names=list(IMPUTERS)
)

Run experiment with scaled train data, store reconstructed dataframes and save reconstruction metrics.

In [6]:
recon_df = experiment.run(train=train_s, store=store)
recon_df.to_parquet(OUTPUTS / "reconstruction_metrics.parquet")


## Results

In [7]:
import pandas as pd
df = pd.read_parquet(OUTPUTS / "reconstruction_metrics.parquet")

In [8]:
df.head()

,mechanism,proportion,seed,imputer,reconstruction_rmse,reconstruction_mae
0,mar,10,0,locf,0.028223,0.017827
1,mar,10,0,ks,0.028223,0.017827
2,mar,10,0,arima,0.028223,0.017827
3,mar,10,0,lstm,0.028223,0.017827
4,mar,30,0,locf,0.033198,0.020716


In [9]:
df.groupby(["imputer", "mechanism", "proportion"]).agg(
    rmse=("reconstruction_rmse", "mean"),
    mae=("reconstruction_mae", "mean")
    )

rmse       mae
imputer mechanism proportion                    
arima   mar       10          0.028688  0.017944
                  30          0.033025  0.020693
                  50          0.044711  0.027979
        mcar      10          0.030199  0.018006
                  30          0.032540  0.019528
                  50          0.036785  0.022114
        mnar      10          0.029911  0.018693
                  30          0.034116  0.021295
                  50          0.045263  0.028225
ks      mar       10          0.028688  0.017944
                  30          0.033025  0.020693
                  50          0.044711  0.027979
        mcar      10          0.030199  0.018006
                  30          0.032540  0.019528
                  50          0.036785  0.022114
        mnar      10          0.029911  0.018693
                  30          0.034116  0.021295
                  50          0.045263  0.028225
locf    mar       10          0.028688  0.017944
                  30          0.033025  0.020693
                  50          0.044711  0.027979
        mcar      10          0.030199  0.018006
                  30          0.032540  0.019528
                  50          0.036785  0.022114
        mnar      10          0.029911  0.018693
                  30          0.034116  0.021295
                  50          0.045263  0.028225
lstm    mar       10          0.028688  0.017944
                  30          0.033025  0.020693
                  50          0.044711  0.027979
        mcar      10          0.030199  0.018006
                  30          0.032540  0.019528
                  50          0.036785  0.022114
        mnar      10          0.029911  0.018693
                  30          0.034116  0.021295
                  50          0.045263  0.028225